In [21]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import io
from tqdm import tqdm
from transformers import AutoProcessor, CLIPModel

In [11]:
test_df = pd.read_csv('/kaggle/input/rsicd-image-caption-dataset/test.csv')
train_df = pd.read_csv('/kaggle/input/rsicd-image-caption-dataset/train.csv')
valid_df = pd.read_csv('/kaggle/input/rsicd-image-caption-dataset/valid.csv')

# IMG_DIR = "/kaggle/input/rsicd-image-caption-dataset"

# valid_df.head(4)


In [22]:
model_name = "openai/clip-vit-base-patch32"  # works without token
processor = AutoProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [23]:
class RSICDDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.data = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load image from bytes
        img_bytes = eval(row["image"])['bytes']
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        # Pick one caption
        caption = eval(row["captions"])[0]

        # Process for CLIP
        inputs = self.processor(
            text=caption,
            images=image,
            return_tensors="pt",
            padding=False,      # padding handled in collate_fn
            truncation=True
        )

        # Remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs


In [24]:
def clip_collate_fn(batch):
    batched = {}
    for key in batch[0]:
        if key == "pixel_values":
            batched[key] = torch.stack([item[key] for item in batch])
        else:
            # pad text sequences to max length in batch
            seqs = [item[key] for item in batch]
            batched[key] = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True)
    return batched

In [25]:
batch_size = 8

valid_dataset = RSICDDataset(valid_df, processor)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=clip_collate_fn
)

In [26]:
def extract_embeddings(data_loader, model, device):
    all_img_embeds = []
    all_txt_embeds = []

    with torch.inference_mode():
        for batch in tqdm(data_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            # normalized embeddings
            img_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
            txt_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

            all_img_embeds.append(img_embeds)
            all_txt_embeds.append(txt_embeds)

    all_img_embeds = torch.cat(all_img_embeds, dim=0)
    all_txt_embeds = torch.cat(all_txt_embeds, dim=0)
    return all_img_embeds, all_txt_embeds

In [27]:
val_img_embeds, val_txt_embeds = extract_embeddings(valid_loader, model, device)
print("Validation embeddings extracted:", val_img_embeds.shape, val_txt_embeds.shape)

100%|██████████| 137/137 [00:09<00:00, 15.10it/s]

Validation embeddings extracted: torch.Size([1094, 512]) torch.Size([1094, 512])


In [28]:
import torch
import torch.nn.functional as F
import pandas as pd

# Compute Recall@K
def compute_recall(img_embeds, txt_embeds, ks=[1,5,10]):
    """
    img_embeds: [N, D]
    txt_embeds: [N, D]
    ks: list of top-K values to evaluate
    """
    # Compute similarity matrix (cosine similarity)
    sim_matrix = img_embeds @ txt_embeds.T  # [N, N]
    
    recalls = {f'R@{k}': 0.0 for k in ks}

    for i in range(sim_matrix.size(0)):
        # Rank text for each image
        sims = sim_matrix[i]
        ranking = torch.argsort(sims, descending=True)  # highest similarity first
        
        for k in ks:
            if i in ranking[:k]:  # correct caption is at position i
                recalls[f'R@{k}'] += 1
    
    # Convert to percentage
    N = sim_matrix.size(0)
    for k in ks:
        recalls[f'R@{k}'] = recalls[f'R@{k}'] * 100.0 / N
    
    return recalls

# Example usage
val_recalls = compute_recall(val_img_embeds, val_txt_embeds)
print("Validation Recall metrics:", val_recalls)

# Format as DataFrame like your example
df = pd.DataFrame({
    'Dataset': ['RSICD'],
    'Image Encoder': ['CLIP'],
    'Knowledge': ['N'],  # You can change this to Y if you apply extra knowledge
    'R@1': [val_recalls['R@1']],
    'R@5': [val_recalls['R@5']],
    'R@10': [val_recalls['R@10']],
})

print(df)


Validation Recall metrics: {'R@1': 4.296160877513711, 'R@5': 14.716636197440584, 'R@10': 24.314442413162705}
  Dataset Image Encoder Knowledge       R@1        R@5       R@10
0   RSICD          CLIP         N  4.296161  14.716636  24.314442


In [29]:
# Extract class name from filename
def get_class_from_filename(filename):
    return filename.split("/")[1].split("_")[0]

train_df['class'] = train_df['filename'].apply(get_class_from_filename)
valid_df['class'] = valid_df['filename'].apply(get_class_from_filename)
test_df['class'] = test_df['filename'].apply(get_class_from_filename)

# Old and new classes
old_classes = set(train_df['class'])
new_classes = set(valid_df['class']) - old_classes
all_classes = set(valid_df['class'])


In [30]:
def compute_recall_by_class(img_embeds, txt_embeds, classes, class_labels):
    """
    img_embeds: tensor of all image embeddings
    txt_embeds: tensor of all text embeddings
    classes: list of unique classes to consider (old/new/all)
    class_labels: list of class names corresponding to each embedding
    """
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    img_embeds = img_embeds / img_embeds.norm(dim=1, keepdim=True)
    txt_embeds = txt_embeds / txt_embeds.norm(dim=1, keepdim=True)
    sims = img_embeds @ txt_embeds.T

    correct = 0
    total = 0
    for i, cls in enumerate(class_labels):
        if cls not in classes:
            continue
        # rank captions by similarity
        ranks = sims[i].argsort(descending=True)
        # check if the correct caption is top-1
        if ranks[0] == i:
            correct += 1
        total += 1

    return 100 * correct / total


In [31]:
# Assume valid_df has 'class' column and embeddings are in same order
class_labels = valid_df['class'].tolist()

all_recall = compute_recall_by_class(val_img_embeds, val_txt_embeds, all_classes, class_labels)
old_recall = compute_recall_by_class(val_img_embeds, val_txt_embeds, old_classes, class_labels)
new_recall = compute_recall_by_class(val_img_embeds, val_txt_embeds, new_classes, class_labels)

print(f"RSICD Image-Text Retrieval Recall@1:")
print(f"All: {all_recall:.2f}  | Old: {old_recall:.2f}  | New: {new_recall:.2f}")


RSICD Image-Text Retrieval Recall@1:
All: 4.30  | Old: 3.99  | New: 8.96
